In [ ]:
#Evaluation Task:
#Design a multi-channel communication strategy
#Suggest how AI could improve language-specific patient communication
#Propose a simple effectiveness measurement approach

In [ ]:
!pip install gtts

In [ ]:
#import required library
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from gtts import gTTS
import IPython.display as ipd
import random
import time

In [ ]:
#translator
def translate(text, model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    translated = model.generate(**inputs)
    return tokenizer.batch_decode(translated, skip_special_tokens=True)[0]

In [ ]:
#speech convertor
def text_to_speech(text,lang):
    tts = gTTS(text=text, lang=lang)
    tts.save("output.mp3")
    ipd.display(ipd.Audio("output.mp3", autoplay=True))

In [ ]:
 models = {
        "Hindi": "Helsinki-NLP/opus-mt-en-hi",
        "Tamil": "suriya7/English-to-Tamil",
        "Malayalam":"Helsinki-NLP/opus-mt-en-ml",
        "Telugu":"facebook/nllb-200-distilled-600M",
        "English":"eng"
    }

In [ ]:
# Sample patient database with language, age, and preferred communication channel
patients = [
    {"id": 1, "name": "Ravi Kumar", "language": "Tamil", "channel": "SMS", "age": 55},
    {"id": 2, "name": "Ananya Rao", "language": "Telugu", "channel": "WhatsApp", "age": 28},
    {"id": 3, "name": "Joseph Mathew", "language": "Malayalam", "channel": "IVR", "age": 65},
    {"id": 4, "name": "Rahul Sharma", "language": "Hindi", "channel": "SMS", "age": 42},
    {"id": 5, "name": "David Thomas", "language": "English", "channel": "WhatsApp", "age": 35},
]

In [ ]:
# message templates for A/B Testing
messages = {
    "A": "your appointment is confirmed. See you soon!" ,
    "B": "our doctor is currently busy. Please wait for some time!"
}

In [ ]:
# Dictionary to store responses and message logs
patient_responses = {}
message_logs = {}

In [ ]:
# Simulate patient response
def confirm_appointment():
    time.sleep(1)  # Simulate delay
    return random.choice(["Confirmed", "Not Confirmed", "No Response"])

In [ ]:
vir_languages = {
    "English": "en",
    "Hindi": "hi",
    "Tamil": "ta",
    "Telugu": "te",
    "Malayalam": "ml"
}

In [ ]:

# Send message based on AI-driven personalization
def send_message(patient, message_type="Appointment Confirmation"):
    response = confirm_appointment()

    # Store response for tracking
    patient_responses[patient["id"]] = response

    #clasifie the message
    message_variant = "A" if response == "Confirmed" else "B"

    #track language
    language_first = patient["language"]

    #get the message basies of it's message_variant
    send_message=messages.get(message_variant)

    #select the model basis of user refer language
    model = models.get(language_first)

    #call translate if the user refer language is not english
    if model!="eng":
      translate_message = translate(send_message,model)
    else:
      translate_message=send_message

    # Modify communication channel if age > 50
    if patient["age"] >50:
      channel = "IVR"
    else:
      channel=patient["channel"]

    if channel =="IVR":
      text_to_speech(translate_message,vir_languages.get(language_first))

    timestamp = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

    # Log the message only if it's the first time the patient gets a response
    if patient["id"] not in message_logs:
        message_logs[patient["id"]] = {
            "patient_id": patient["id"],
            "name": patient["name"],
            "language": language_first,
            "channel": channel,
            "message_type": message_type,
            "message": translate_message,
            "timestamp": timestamp,
            "patient_status": response
        }

    print(f"\n📩 [{timestamp}] {message_type} (Variant {message_variant}) via {channel} to {patient['name']} ({language_first}): {translate_message}")

In [ ]:
# Simulate messaging for all patients
for patient in patients:
    send_message(patient, "Appointment Confirmation")
    time.sleep(1)


📩 [2025-03-29 11:44:20] Appointment Confirmation (Variant A) via IVR to Ravi Kumar (Tamil): விரைவில் பார்த்துக்கொள்ளுங்கள்!

📩 [2025-03-29 11:44:46] Appointment Confirmation (Variant B) via WhatsApp to Ananya Rao (Telugu): ¡Por favor, espere un tiempo!



📩 [2025-03-29 11:44:52] Appointment Confirmation (Variant B) via IVR to Joseph Mathew (Malayalam): ഡോക്റ്റർ ഇപ്പോൾ തിരക്കിലാണ്, കുറച്ചു സമയം കാത്തിരിക്കൂ!

📩 [2025-03-29 11:44:57] Appointment Confirmation (Variant A) via SMS to Rahul Sharma (Hindi): तुम्हारी नियुक्‍ति पक्की है ।

📩 [2025-03-29 11:44:59] Appointment Confirmation (Variant B) via WhatsApp to David Thomas (English): our doctor is currently busy. Please wait for some time!


In [ ]:
# Analyze confirmation rate and provide future recommendations
def measure_effectiveness():
    confirmed_count=0
    for log_id, details in message_logs.items():
      if details['patient_status']== "Confirmed":
          confirmed_count +=1
      total_sent =len(patients)
    print(confirmed_count)

    print("\n Patient Responses & Message Logs:")
    for log in message_logs.values():
        print(f"{log['name']} ({log['language']}) - {log['patient_status']} via {log['channel']}")

    confirmation_rate = (confirmed_count / total_sent) * 100
    print(f"\n Overall Confirmation Rate: {confirmation_rate:.2f}%")
measure_effectiveness()

3

 Patient Responses & Message Logs:
Ravi Kumar (Tamil) - Not Confirmed via IVR
Ananya Rao (Telugu) - Confirmed via WhatsApp
Joseph Mathew (Malayalam) - No Response via IVR
Rahul Sharma (Hindi) - Confirmed via SMS
David Thomas (English) - Confirmed via WhatsApp

 Overall Confirmation Rate: 60.00%


In [ ]:
!pip install sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 16.8 MB/s eta 0:00:00


In [ ]:
message_logs

{1: {'patient_id': 1,
  'name': 'Ravi Kumar',
  'language': 'Tamil',
  'channel': 'IVR',
  'message_type': 'Appointment Confirmation',
  'message': 'எங்களது டாக்டர் பிஸி.',
  'timestamp': '2025-03-29 09:44:17',
  'patient_status': 'Not Confirmed'},
 2: {'patient_id': 2,
  'name': 'Ananya Rao',
  'language': 'Telugu',
  'channel': 'WhatsApp',
  'message_type': 'Appointment Confirmation',
  'message': '¡Por favor, espere un tiempo!',
  'timestamp': '2025-03-29 09:44:50',
  'patient_status': 'No Response'},
 3: {'patient_id': 3,
  'name': 'Joseph Mathew',
  'language': 'Malayalam',
  'channel': 'IVR',
  'message_type': 'Appointment Confirmation',
  'message': 'ഡോക്റ്റർ ഇപ്പോൾ തിരക്കിലാണ്, കുറച്ചു സമയം കാത്തിരിക്കൂ!',
  'timestamp': '2025-03-29 09:45:09',
  'patient_status': 'No Response'},
 4: {'patient_id': 4,
  'name': 'Rahul Sharma',
  'language': 'Hindi',
  'channel': 'SMS',
  'message_type': 'Appointment Confirmation',
  'message': 'हमारे डॉक्टर अभी व्यस्त हैं. कृपया कुछ समय के लिए इ